## this notebook is used to plot cohort concordance and GWAS results from the "Cohort Builder"-derived cohort and SQL-derived cohort

adapted from notebook:

"5 - GWAS Figures 2024.ipynb"

in workspace "Hypothyroidism genomics v7"

In [ ]:
import pandas as pd
import numpy as np
import os 
import subprocess
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import json
import re
import gcsfs
plt.rcParams['figure.dpi'] = 300

In [ ]:
def wb(*args):
    """Run a wb command and return parsed JSON."""
    cmd = ["wb", *args, "--format=json"]
    result = subprocess.check_output(cmd, text=True)
    return json.loads(result)


# Get workspace info
workspace = wb("workspace", "describe")
google_project_id = workspace["googleProjectId"]

# Get resources
resources = wb("resource", "list")

# WORKSPACE_BUCKET
bucket_resources = [
    r for r in resources
    if r.get("resourceType") == "GCS_BUCKET"
    and "practical_considerations_bucket" in r.get("id", "")
    and "temporary" not in r.get("id", "")
]

if not bucket_resources:
    raise ValueError("No matching bucket found")

bucket = f"gs://{bucket_resources[0]['bucketName']}"

# WORKSPACE_CDR
bq_resources = [
    r for r in resources
    if r.get("resourceType") in {"BQ_DATASET", "BIGQUERY_DATASET"}
]

cdr_resources = [
    r for r in bq_resources
    if re.match(r"^C\d{4}Q\d+R\d+$", r.get("datasetId", ""))
]

if not cdr_resources:
    raise ValueError("No matching CDR dataset found")

CDR = (
    f"{cdr_resources[0]['projectId']}."
    f"{cdr_resources[0]['datasetId']}"
)


In [ ]:
fs = gcsfs.GCSFileSystem(requester_pays=True)
with fs.open(f'{bucket}/hypothyroid_data/cb_v2_phenotype_covars.tsv') as f:
    cb = pd.read_csv(f, sep='\t')

In [ ]:
with fs.open(f'{bucket}/hypothyroid_data/huan_phenotype_v4_covars.csv') as f:
    hu = pd.read_csv(f)

In [ ]:
cb_0 = cb[cb['Hypothyroidism']==0].person_id.unique()
cb_1 = cb[cb['Hypothyroidism']==1].person_id.unique()
hu_0 = hu[hu['Hypothyroidism']==0].person_id.unique()
hu_1 = hu[hu['Hypothyroidism']==1].person_id.unique()

In [ ]:
caseoverlap = venn2([set(cb_1), set(hu_1)], set_labels=('Cohort Builder','SQL'))

In [ ]:
venn2([set(cb_0), set(hu_0)], set_labels=('Cohort Builder','BigQuery/SQL'))

### Plot correlation of cb and sql gwas

In [ ]:
#read in gwas results from cb and sql
cb_tra = pd.read_csv(f'{bucket}/hypothyroid_data/cb_gwas.tsv.bgz', sep = '\t', compression = 'gzip')
hu_tra = pd.read_csv(f'{bucket}/hypothyroid_data/hu_gwas.tsv.bgz', sep = '\t', compression = 'gzip')

cb_tra.head()
hu_tra.head()

In [ ]:
cb_tra = cb_tra.dropna()
hu_tra = hu_tra.dropna()

In [ ]:
merged = cb_tra[['locus', 'alleles', 'beta', 'p_value']].merge(
    hu_tra[['locus', 'alleles', 'beta', 'p_value']],
    on = ['locus', 'alleles'],
    how = 'left',
    suffixes = ('_cb', '_hu')
)
merged

In [ ]:
limit = 1*10**-5
merged_cb = merged[
    (merged.beta_cb.between(-2,2)) & (merged.beta_hu.between(-2,2)) &
    (merged.p_value_cb < limit)
]

merged_all = merged[
    (merged.beta_cb.between(-2,2)) & (merged.beta_hu.between(-2,2)) &
    ((merged.p_value_cb < limit) | (merged.p_value_hu < limit))
]

In [ ]:
from scipy.stats import linregress


In [ ]:
def plot_correlation(df, x, y, frac = 1, xlabel='X', ylabel='Y', title='Correlation Plot'):
    # Ensure x and y are numpy arrays
    df = df.sample(frac= frac)
    x = np.array(df[x])
    y = np.array(df[y])
    
    # Create the scatter plot
    plt.figure(figsize=(4, 4))
    plt.plot(x, y, alpha=0.5, marker='.', linestyle='None')
    
    # Calculate the linear regression
    slope, intercept, r_value, p_value, std_err = linregress(x, y)
    line = slope * x + intercept
    
    # Plot the regression line
    plt.plot(x, line, linestyle = '-', color='black', alpha = 0.5, label=f'y = {slope:.2f}x + {intercept:.2f}')
    
    # Calculate R-squared
    r_squared = r_value**2
    
    # Add labels and title
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    
    # Add R-squared value to the plot
    plt.text(0.05, 0.85, f'R² = {r_squared:.3f}', transform=plt.gca().transAxes,
             verticalalignment='top')
    
    # Add legend
    plt.legend()
    
    # Show the plot
    plt.tight_layout()
    plt.show()

In [ ]:
#correlation of nominally sig snps in CB
plot_correlation(merged_cb, 'beta_cb', 'beta_hu', xlabel = 'Cohort Builder', ylabel = 'BigQuery/SQL', title='Beta comparison of nominally significant SNPs')

In [ ]:
#correlation of nominally significant snps in either cb or sql
plot_correlation(merged_all, 'beta_cb', 'beta_hu', xlabel = 'Cohort Builder', ylabel = 'BigQuery/SQL', title='Beta comparison of nominally significant SNPs')

In [ ]:
len(merged_cb)

In [ ]:
len(merged_all)

### gwas manhattan plots

In [ ]:
CHROM_COLORS     = ["#222222", "#AAAAAA"]   # alternating black / grey
SUGGESTIVE_COLOR = "#2196F3"                # blue  dashed  – p = 1e-5
GWSIG_COLOR      = "#E53935"                # red   solid   – p = 5e-8


def parse_locus(locus: str):
    chrom_str, pos_str = locus.split(":")
    chrom_str = chrom_str.upper().replace("CHR", "").lower()
    chrom_map = {"x": 23, "y": 24, "mt": 25, "m": 25}
    chrom = chrom_map[chrom_str] if chrom_str in chrom_map else int(chrom_str)
    return chrom, int(pos_str)


def manhattan_plot(
    df: pd.DataFrame,
    locus_col: str    = "locus",
    pval_col: str     = "p_value",
    suggestive: float = 1e-5,
    gwsig: float      = 5e-8,
    title: str        = "Manhattan Plot",
    point_size: float = 8,
    alpha: float      = 0.8,
    save_path: str    = None,
) -> plt.Figure:
    parsed        = df[locus_col].apply(parse_locus)
    work          = df[[locus_col, pval_col]].copy()
    work["chrom"] = parsed.apply(lambda x: x[0])
    work["pos"]   = parsed.apply(lambda x: x[1])

    work = work[(work[pval_col] > 0) & work[pval_col].notna()].copy()
    work["-log10p"] = -np.log10(work[pval_col].astype(float))

    chroms = sorted(work["chrom"].unique())
    running = 0
    x_tick_centers, x_tick_labels = [], []

    for chrom in chroms:
        mask    = work["chrom"] == chrom
        min_pos = work.loc[mask, "pos"].min()
        max_pos = work.loc[mask, "pos"].max()
        # set x_global so chromosome spans [running, running + span]
        work.loc[mask, "x_global"] = work.loc[mask, "pos"] - min_pos + running
        span = max_pos - min_pos
        # center should be running + span/2
        x_tick_centers.append(running + span / 2)
        label = str(chrom) if chrom <= 22 else {23: "X", 24: "Y", 25: "MT"}[chrom]
        x_tick_labels.append(label)
        running += span + int(5e6)

    # larger width to accommodate bigger labels
    fig, ax = plt.subplots(figsize=(10, 10))

    for i, chrom in enumerate(chroms):
        mask  = work["chrom"] == chrom
        color = CHROM_COLORS[i % 2]
        ax.scatter(
            work.loc[mask, "x_global"],
            work.loc[mask, "-log10p"],
            c=color,
            s=point_size,
            alpha=alpha,
            linewidths=0,
            rasterized=True,
        )

    sug_y = -np.log10(suggestive)
    gws_y = -np.log10(gwsig)
    ymax = 80

    ax.axhline(sug_y, color=SUGGESTIVE_COLOR, linewidth=1.2, linestyle="--",
               zorder=3, label=f"Suggestive  (p = {suggestive:.0e})")
    ax.axhline(gws_y, color=GWSIG_COLOR,      linewidth=1.4, linestyle="-",
               zorder=3, label=f"Genome-wide (p = {gwsig:.0e})")

    # Font sizes
    xtick_fs = 14
    ytick_fs = 14
    label_fs = 16
    title_fs = 18

    ax.set_xlim(work["x_global"].min(), work["x_global"].max())
    ax.set_ylim(0, ymax)
    ax.set_xticks(x_tick_centers)
    # Center labels on ticks to avoid the slight offset; use va='top' when rotated so label anchors neatly
    ax.set_xticklabels(x_tick_labels, fontsize=xtick_fs, rotation=90, ha="center", va="top")
    ax.tick_params(axis="y", labelsize=ytick_fs)

    ax.set_xlabel("Chromosome", fontsize=label_fs, labelpad=10)
    ax.set_ylabel(r"$-\log_{10}(p)$", fontsize=label_fs, labelpad=10)
    ax.set_title(title, fontsize=title_fs, fontweight="bold", pad=12)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=False, fontsize=12, loc="upper right")

    plt.tight_layout()
    fig.subplots_adjust(bottom=0.20)

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return fig


In [ ]:
manhattan_plot(cb_tra[cb_tra.locus.str.contains("X|Y")==False], title="CB Manhattan Plot")

In [ ]:
manhattan_plot(hu_tra[hu_tra.locus.str.contains("X|Y")==False], title="SQL Manhattan Plot")

### find number of significant peaks

In [ ]:
from scipy import signal
import logging

In [ ]:
def find_peaks(gwas, only_significant = True, prom = 6, dist = 10000):
    gwas = gwas[gwas['fit.converged']==True]
    gwas_array = -np.log10(np.array(gwas.p_value))
    peaks = signal.find_peaks(gwas_array, prominence = prom, distance = dist)[0]
    peak_hits = gwas.iloc[list(peaks)]
    
    # count hits sig or not
    print('# of significant peaks: ' + str(len(peak_hits[peak_hits.p_value<=5*10**-8])))
    print('# of nonsignificant peaks: ' + str(len(peak_hits[peak_hits.p_value>5*10**-8])))
    
    # plot 
    logging.info('Identification complete, plotting')
    colo = []
    for i in gwas_array[peaks]:
        if i > -np.log10(5*10**-8):
            colo.append('red')
        else:
            colo.append('tomato')

    plt.plot(gwas_array, color = 'lightsteelblue')
    plt.scatter(peaks,gwas_array[peaks],marker='x',color=colo)
    plt.axhline(y=-np.log10(5*10**-8),color='cornflowerblue',linestyle='-')
    
    # return only signfiicant hits if asked
    if only_significant == True:
        peak_hits = peak_hits[peak_hits.p_value < 5*10**-8]
    return peak_hits

In [ ]:
find_peaks(cb_tra)

In [ ]:
find_peaks(hu_tra)